# NB53 — Counterfactual Response Fingerprint
Standalone inference-only pipeline using only competition data and the provided poisoned model.
It validates a new per-box causal response signal, continuously rescales every candidate, and never
loads artifacts from earlier notebooks. Primary output: `nb53_CF_BALANCED.csv`.


In [ ]:
import subprocess

def sh(command):
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print((result.stdout + result.stderr)[-3000:])
        raise RuntimeError(command)

sh("pip install -q 'setuptools<81'")
sh("pip install -q --no-cache-dir 'git+https://github.com/facebookresearch/detectron2.git'")

import torch
cuda_value = float((torch.ones(8, device="cuda") * 2).sum().item())
print(f"setup | gpu={torch.cuda.get_device_name(0)} torch={torch.__version__} cuda={cuda_value:.0f}")


In [ ]:
import glob
import hashlib
import json
import logging
import os
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score, roc_curve

from detectron2 import model_zoo
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.config import get_cfg
from detectron2.data import transforms as T
from detectron2.modeling import build_model

for logger_name in ("detectron2", "fvcore", "d2"):
    logging.getLogger(logger_name).setLevel(logging.ERROR)

MODE = "standalone_counterfactual_response"
CFG = dict(
    SEED=42,
    N_FOLDS=4,
    CAND_THRESH=0.02,
    BASE_THR=0.10,
    AMP_INFER=True,
    MATCH_IOU=0.20,
    ANCHOR_MIN_SCORE=0.45,
    ANCHOR_FLIP_MIN_SCORE=0.35,
    ANCHOR_MIN_IOU=0.70,
    MIN_REAL_ANCHORS=64,
    TARGET_REAL_ANCHORS=96,
    ANCHOR_POOL_IMAGES=700,
    RESPONSE_AUC_MIN=0.85,
    RESPONSE_PARTIAL_AUC_MIN=0.72,
    RESPONSE_TPR10_MIN=0.55,
    RESPONSE_FOLD_AUC_MIN=0.65,
    RESPONSE_MARGIN_MIN=0.08,
    EPS=0.01,
    INFER_BATCH=3,
)
RESPONSE_CANDIDATES = [
    {"name": "CF_BALANCED", "strength": 0.78, "power": 1.50, "priority": 1},
    {"name": "CF_CONSERVATIVE", "strength": 0.62, "power": 1.75, "priority": 2},
    {"name": "CF_AGGRESSIVE", "strength": 0.90, "power": 1.25, "priority": 3},
]
assert sum(candidate["priority"] == 1 for candidate in RESPONSE_CANDIDATES) == 1
OUTPUT_BY_NAME = {
    "CF_BALANCED": "nb53_CF_BALANCED.csv",
    "CF_CONSERVATIVE": "nb53_CF_CONSERVATIVE.csv",
    "CF_AGGRESSIVE": "nb53_CF_AGGRESSIVE.csv",
}
FEATURE_NAMES = [
    "removal_drop", "amplitude_35", "amplitude_65", "amplitude_curvature",
    "low_frequency_only", "high_frequency_only", "frequency_interaction",
    "baseline_score",
]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WORK = Path("/kaggle/working")
WORK.mkdir(parents=True, exist_ok=True)
for pattern in ("nb53_*.csv", "nb53_*.json"):
    for stale_path in WORK.glob(pattern):
        stale_path.unlink(missing_ok=True)

np.random.seed(CFG["SEED"])
torch.manual_seed(CFG["SEED"])
print(f"config | mode={MODE} device={DEVICE} candidates={len(RESPONSE_CANDIDATES)}")


In [ ]:
def resolve_competition_data():
    roots = [
        Path("/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models"),
        Path("/kaggle/input/neural-debris-removal-in-streak-detection-models"),
    ]
    data_root = next((root for root in roots if root.exists()), None)
    if data_root is None:
        hits = sorted(glob.glob("/kaggle/input/**/annotations_coco.json", recursive=True))
        assert hits, "competition data not found"
        data_root = Path(hits[0]).parents[1]

    annotation_hits = sorted(data_root.glob("**/annotations_coco.json"))
    sample_hits = sorted(data_root.glob("**/sample_submission.csv"))
    weight_hits = sorted(data_root.glob("**/*.pth"))
    assert annotation_hits and sample_hits and weight_hits
    poisoned_weights = next(
        (path for path in weight_hits if "poison" in path.name.lower()), weight_hits[0]
    )
    png_by_parent = {}
    for path in data_root.glob("**/*.png"):
        png_by_parent.setdefault(path.parent, []).append(path)
    test_dir = max(png_by_parent, key=lambda path: len(png_by_parent[path]))
    return {
        "root": data_root,
        "annotations": annotation_hits[0],
        "unlearn_dir": annotation_hits[0].parent,
        "sample": sample_hits[0],
        "test_dir": test_dir,
        "weights": poisoned_weights,
    }

DATA = resolve_competition_data()
UNLEARN_JSON = DATA["annotations"]
UNLEARN_DIR = DATA["unlearn_dir"]
SAMPLE_SUB = DATA["sample"]
TEST_DIR = DATA["test_dir"]
POISONED_WEIGHTS = DATA["weights"]

BASE_CONFIG = "COCO-Detection/retinanet_R_50_FPN_3x.yaml"
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES = [[16], [32], [64], [128], [256]]

def load_image(path):
    image = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    assert image is not None, path
    if image.dtype == np.uint16:
        image = image.astype(np.float32) / 65535.0
    else:
        image = image.astype(np.float32) / 255.0
    if image.ndim == 2:
        image = np.repeat(image[:, :, None], 3, axis=2)
    return np.clip(image * 255.0, 0, 255).astype(np.float32)

def build_cfg(weights):
    config = get_cfg()
    config.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    config.MODEL.WEIGHTS = str(weights)
    config.MODEL.RETINANET.NUM_CLASSES = 1
    config.MODEL.RETINANET.SCORE_THRESH_TEST = CFG["CAND_THRESH"]
    config.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
    config.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    config.MODEL.DEVICE = DEVICE
    return config

def load_model(weights=POISONED_WEIGHTS):
    model = build_model(build_cfg(weights))
    DetectionCheckpointer(model).load(str(weights))
    model.to(DEVICE).eval()
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    return model

resize_aug = T.ResizeShortestEdge(short_edge_length=800, max_size=1333)

def to_batched(image):
    height, width = image.shape[:2]
    transform = resize_aug.get_transform(image)
    resized = transform.apply_image(image)
    return {
        "image": torch.as_tensor(np.ascontiguousarray(resized.transpose(2, 0, 1))),
        "height": height,
        "width": width,
    }

@torch.no_grad()
def infer_many(model, images, batch_size=None):
    batch_size = batch_size or CFG["INFER_BATCH"]
    predictions = []
    for start in range(0, len(images), batch_size):
        records = [to_batched(image) for image in images[start:start + batch_size]]
        with torch.autocast("cuda", enabled=(CFG["AMP_INFER"] and DEVICE == "cuda")):
            outputs = model(records)
        for output in outputs:
            instances = output["instances"].to("cpu")
            predictions.append((
                instances.pred_boxes.tensor.numpy().astype(np.float32),
                instances.scores.numpy().astype(np.float32),
            ))
    return predictions

def infer_np(model, image):
    return infer_many(model, [image], batch_size=1)[0]

def iou_matrix(first, second):
    first = np.asarray(first, np.float32).reshape(-1, 4)
    second = np.asarray(second, np.float32).reshape(-1, 4)
    if len(first) == 0 or len(second) == 0:
        return np.zeros((len(first), len(second)), np.float32)
    left = np.maximum(first[:, None, :2], second[None, :, :2])
    right = np.minimum(first[:, None, 2:], second[None, :, 2:])
    size = np.maximum(0.0, right - left)
    intersection = size[:, :, 0] * size[:, :, 1]
    first_area = np.prod(np.maximum(0.0, first[:, 2:] - first[:, :2]), axis=1)[:, None]
    second_area = np.prod(np.maximum(0.0, second[:, 2:] - second[:, :2]), axis=1)[None, :]
    union = first_area + second_area - intersection
    return np.where(union > 0, intersection / union, 0.0).astype(np.float32)

def match_score(target_box, boxes, scores, minimum_iou=None):
    minimum_iou = minimum_iou or CFG["MATCH_IOU"]
    if len(boxes) == 0:
        return 0.0
    overlaps = iou_matrix(np.asarray(target_box).reshape(1, 4), boxes)[0]
    valid = overlaps >= minimum_iou
    return float(scores[valid].max()) if valid.any() else 0.0

model = load_model()
test_files = sorted(TEST_DIR.glob("*.png"), key=lambda path: int(path.stem))
print(f"data | root={DATA['root']} test={len(test_files)} weights={POISONED_WEIGHTS.name}")


In [ ]:
def clip_box(box, width, height, margin=0.10):
    x1, y1, x2, y2 = [float(value) for value in box]
    margin_x = margin * max(x2 - x1, 1.0)
    margin_y = margin * max(y2 - y1, 1.0)
    return (
        int(np.clip(np.floor(x1 - margin_x), 0, width - 1)),
        int(np.clip(np.floor(y1 - margin_y), 0, height - 1)),
        int(np.clip(np.ceil(x2 + margin_x), 1, width)),
        int(np.clip(np.ceil(y2 + margin_y), 1, height)),
    )

def inpaint_box(image, box):
    height, width = image.shape[:2]
    x1, y1, x2, y2 = clip_box(box, width, height)
    mask = np.zeros((height, width), np.uint8)
    mask[y1:y2, x1:x2] = 255
    uint8_image = np.clip(image, 0, 255).astype(np.uint8)
    return cv2.inpaint(uint8_image, mask, 5, cv2.INPAINT_TELEA).astype(np.float32)

def replace_region(image, region, values):
    output = image.copy()
    x1, y1, x2, y2 = region
    output[y1:y2, x1:x2] = np.clip(values, 0, 255)
    return output.astype(np.float32)

def build_counterfactual_stack(image, box):
    height, width = image.shape[:2]
    region = clip_box(box, width, height)
    x1, y1, x2, y2 = region
    inpainted = inpaint_box(image, box)
    original_region = image[y1:y2, x1:x2]
    background_region = inpainted[y1:y2, x1:x2]
    signal = original_region - background_region
    sigma = max(1.0, min(y2 - y1, x2 - x1) / 10.0)
    low_frequency = cv2.GaussianBlur(signal, (0, 0), sigmaX=sigma, sigmaY=sigma)
    high_frequency = signal - low_frequency
    return {
        "inpainted": inpainted,
        "amplitude_35": replace_region(image, region, background_region + 0.35 * signal),
        "amplitude_65": replace_region(image, region, background_region + 0.65 * signal),
        "low_frequency_only": replace_region(image, region, background_region + low_frequency),
        "high_frequency_only": replace_region(image, region, background_region + high_frequency),
    }

def response_fingerprint(base_score, variant_scores):
    divisor = max(float(base_score), 1e-4)
    removal_ratio = np.clip(variant_scores["inpainted"] / divisor, 0.0, 1.5)
    amplitude_35 = np.clip(variant_scores["amplitude_35"] / divisor, 0.0, 1.5)
    amplitude_65 = np.clip(variant_scores["amplitude_65"] / divisor, 0.0, 1.5)
    low_frequency_only = np.clip(
        variant_scores["low_frequency_only"] / divisor, 0.0, 1.5
    )
    high_frequency_only = np.clip(
        variant_scores["high_frequency_only"] / divisor, 0.0, 1.5
    )
    frequency_interaction = 1.0 - max(low_frequency_only, high_frequency_only)
    values = np.asarray([
        1.0 - removal_ratio,
        amplitude_35,
        amplitude_65,
        amplitude_65 - amplitude_35,
        low_frequency_only,
        high_frequency_only,
        frequency_interaction,
        float(base_score),
    ], np.float32)
    return dict(zip(FEATURE_NAMES, values.tolist()))

def fingerprint_candidate(model, image, target_box, base_score):
    stack = build_counterfactual_stack(image, target_box)
    names = list(stack)
    predictions = infer_many(model, [stack[name] for name in names])
    variant_scores = {
        name: match_score(target_box, boxes, scores)
        for name, (boxes, scores) in zip(names, predictions)
    }
    return response_fingerprint(base_score, variant_scores)

with open(UNLEARN_JSON, "r") as handle:
    annotations = json.load(handle)
filename_by_id = {image["id"]: image["file_name"] for image in annotations["images"]}
poison_records = []
for annotation in annotations["annotations"]:
    filename = filename_by_id[annotation["image_id"]]
    path = UNLEARN_DIR / filename
    x, y, width, height = annotation["bbox"]
    poison_records.append({
        "source_id": Path(filename).stem,
        "path": path,
        "ground_truth": np.asarray([x, y, x + width, y + height], np.float32),
    })
poison_source_ids = {record["source_id"] for record in poison_records}
print(f"poison | records={len(poison_records)} sources={len(poison_source_ids)}")


In [ ]:
def split_source_folds(source_ids, number_of_folds, seed):
    identities = sorted(source_ids)
    rng = np.random.default_rng(seed)
    rng.shuffle(identities)
    folds = [[] for _ in range(number_of_folds)]
    for index, identity in enumerate(identities):
        folds[index % number_of_folds].append(identity)
    assert all(folds)
    return folds

SOURCE_FOLDS = split_source_folds(poison_source_ids, CFG["N_FOLDS"], CFG["SEED"])

def unflip_boxes(boxes, width):
    output = boxes.copy()
    output[:, 0] = width - boxes[:, 2]
    output[:, 2] = width - boxes[:, 0]
    return output

def mine_stable_real_anchors(model, paths, poison_source_ids):
    anchors = []
    if len(paths) > CFG["ANCHOR_POOL_IMAGES"]:
        indices = np.linspace(
            0, len(paths) - 1, CFG["ANCHOR_POOL_IMAGES"], dtype=np.int64
        )
        paths = [paths[index] for index in indices]
    for path in paths:
        source_id = path.stem
        if source_id not in poison_source_ids:
            image = load_image(path)
            flipped = np.ascontiguousarray(image[:, ::-1])
            (boxes, scores), (flip_boxes, flip_scores) = infer_many(
                model, [image, flipped], batch_size=2
            )
            flip_boxes = unflip_boxes(flip_boxes, image.shape[1])
            eligible = np.flatnonzero(scores >= CFG["ANCHOR_MIN_SCORE"])
            if len(eligible) and len(flip_boxes):
                overlaps = iou_matrix(boxes[eligible], flip_boxes)
                best_flip = overlaps.argmax(axis=1)
                stable = (
                    (overlaps.max(axis=1) >= CFG["ANCHOR_MIN_IOU"])
                    & (flip_scores[best_flip] >= CFG["ANCHOR_FLIP_MIN_SCORE"])
                )
                stable_indices = eligible[stable]
                if len(stable_indices):
                    best = stable_indices[np.argmax(scores[stable_indices])]
                    anchors.append({
                        "source_id": source_id,
                        "path": path,
                        "box": boxes[best].copy(),
                        "score": float(scores[best]),
                    })
        if len(anchors) >= CFG["TARGET_REAL_ANCHORS"]:
            break
    assert len(anchors) >= CFG["MIN_REAL_ANCHORS"], len(anchors)
    return anchors

real_anchors = mine_stable_real_anchors(model, test_files, poison_source_ids)
print(f"anchors | count={len(real_anchors)} folds={[len(fold) for fold in SOURCE_FOLDS]}")


In [ ]:
def positive_response_rows(model, records):
    rows = []
    for record in records:
        image = load_image(record["path"])
        boxes, scores = infer_np(model, image)
        overlaps = iou_matrix(record["ground_truth"].reshape(1, 4), boxes)[0]
        if len(overlaps) == 0:
            continue
        index = int(overlaps.argmax())
        if overlaps[index] < CFG["MATCH_IOU"]:
            continue
        fingerprint = fingerprint_candidate(model, image, boxes[index], scores[index])
        rows.append({
            "source_id": record["source_id"],
            "label": 1,
            **fingerprint,
        })
    return rows

def negative_response_rows(model, anchors):
    rows = []
    for anchor in anchors:
        image = load_image(anchor["path"])
        fingerprint = fingerprint_candidate(
            model, image, anchor["box"], anchor["score"]
        )
        rows.append({
            "source_id": anchor["source_id"],
            "label": 0,
            **fingerprint,
        })
    return rows

response_frame = pd.DataFrame(
    positive_response_rows(model, poison_records)
    + negative_response_rows(model, real_anchors)
)
assert response_frame["label"].sum() >= 16
assert (response_frame["label"] == 0).sum() >= CFG["MIN_REAL_ANCHORS"]
print(
    f"responses | poison={int(response_frame['label'].sum())} "
    f"real={int((response_frame['label'] == 0).sum())}"
)


In [ ]:
def fit_robust_calibrator(frame):
    features = frame[FEATURE_NAMES].to_numpy(np.float64)
    labels = frame["label"].to_numpy(np.int64)
    positive = features[labels == 1]
    negative = features[labels == 0]
    positive_median = np.median(positive, axis=0)
    negative_median = np.median(negative, axis=0)
    center = 0.5 * (positive_median + negative_median)
    pooled = np.vstack([positive, negative])
    scale = np.median(np.abs(pooled - np.median(pooled, axis=0)), axis=0)
    scale = np.maximum(scale, 0.03)
    direction = (positive_median - negative_median) / scale
    direction = direction / max(np.linalg.norm(direction), 1e-6)
    raw = ((features - center) / scale) @ direction
    positive_raw = raw[labels == 1]
    negative_raw = raw[labels == 0]
    raw_center = 0.5 * (np.median(positive_raw) + np.median(negative_raw))
    raw_scale = max(
        0.5 * abs(np.median(positive_raw) - np.median(negative_raw)), 0.20
    )
    return {
        "center": center.tolist(),
        "scale": scale.tolist(),
        "direction": direction.tolist(),
        "raw_center": float(raw_center),
        "raw_scale": float(raw_scale),
    }

def score_fingerprint(features, calibrator):
    features = np.asarray(features, np.float64)
    center = np.asarray(calibrator["center"], np.float64)
    scale = np.asarray(calibrator["scale"], np.float64)
    direction = np.asarray(calibrator["direction"], np.float64)
    raw = ((features - center) / scale) @ direction
    logits = (raw - calibrator["raw_center"]) / calibrator["raw_scale"]
    return 1.0 / (1.0 + np.exp(-np.clip(logits, -12, 12)))

def partial_auc(labels, scores, maximum_fpr=0.10):
    false_positive_rate, true_positive_rate, _ = roc_curve(labels, scores)
    crossing = int(np.searchsorted(false_positive_rate, maximum_fpr, side="right"))
    x_values = false_positive_rate[:crossing]
    y_values = true_positive_rate[:crossing]
    if len(x_values) == 0 or x_values[-1] < maximum_fpr:
        right = min(crossing, len(false_positive_rate) - 1)
        left = max(0, right - 1)
        denominator = false_positive_rate[right] - false_positive_rate[left]
        weight = 0.0 if denominator <= 0 else (
            maximum_fpr - false_positive_rate[left]
        ) / denominator
        interpolated = true_positive_rate[left] + weight * (
            true_positive_rate[right] - true_positive_rate[left]
        )
        x_values = np.append(x_values, maximum_fpr)
        y_values = np.append(y_values, interpolated)
    return float(np.trapezoid(y_values, x_values) / maximum_fpr)

def tpr_at_fpr(labels, scores, maximum_fpr=0.10):
    false_positive_rate, true_positive_rate, _ = roc_curve(labels, scores)
    valid = false_positive_rate <= maximum_fpr + 1e-12
    return float(true_positive_rate[valid].max()) if valid.any() else 0.0

def crossfit_response_audit(frame):
    rows = []
    fold_auc = []
    fold_by_source = {
        source_id: fold_index
        for fold_index, fold in enumerate(SOURCE_FOLDS)
        for source_id in fold
    }
    negative_sources = sorted(frame.loc[frame["label"] == 0, "source_id"].unique())
    for source_id in negative_sources:
        digest = hashlib.sha1(f"negative:{CFG['SEED']}:{source_id}".encode()).hexdigest()
        fold_by_source[source_id] = int(digest[:8], 16) % CFG["N_FOLDS"]
    working = frame.copy()
    working["fold"] = working["source_id"].map(fold_by_source).astype(int)
    for fold_index in range(CFG["N_FOLDS"]):
        training = working[working["fold"] != fold_index]
        held = working[working["fold"] == fold_index]
        calibrator = fit_robust_calibrator(training)
        held_features = held[FEATURE_NAMES].to_numpy(np.float64)
        held_scores = score_fingerprint(held_features, calibrator)
        fold_labels = held["label"].to_numpy(np.int64)
        auc = float(roc_auc_score(fold_labels, held_scores))
        fold_auc.append(auc)
        for row_index, score in zip(held.index, held_scores):
            rows.append({
                "row_index": int(row_index),
                "fold": fold_index,
                "label": int(working.loc[row_index, "label"]),
                "source_id": working.loc[row_index, "source_id"],
                "poison_probability": float(score),
            })
    return pd.DataFrame(rows).sort_values("row_index"), fold_auc

oof_response, fold_auc = crossfit_response_audit(response_frame)
labels = oof_response["label"].to_numpy(np.int64)
scores = oof_response["poison_probability"].to_numpy(np.float64)
response_auc = float(roc_auc_score(labels, scores))
response_partial_auc = partial_auc(labels, scores)
response_tpr10 = tpr_at_fpr(labels, scores)
response_margin = float(
    np.median(scores[labels == 1]) - np.median(scores[labels == 0])
)
RESPONSE_GATES_PASS = bool(
    response_auc >= CFG["RESPONSE_AUC_MIN"]
    and response_partial_auc >= CFG["RESPONSE_PARTIAL_AUC_MIN"]
    and response_tpr10 >= CFG["RESPONSE_TPR10_MIN"]
    and min(fold_auc) >= CFG["RESPONSE_FOLD_AUC_MIN"]
    and response_margin >= CFG["RESPONSE_MARGIN_MIN"]
)
oof_response.to_csv(WORK / "nb53_response_oof.csv", index=False)
full_calibrator = fit_robust_calibrator(response_frame)
print("\n=== NB53 RESPONSE AUDIT ===")
print(
    f"auc={response_auc:.4f} pauc10={response_partial_auc:.4f} "
    f"tpr10={response_tpr10:.4f} margin={response_margin:.4f} "
    f"folds={[round(value, 4) for value in fold_auc]} pass={RESPONSE_GATES_PASS}"
)
if not RESPONSE_GATES_PASS:
    raise RuntimeError("NB53 fail closed: counterfactual response did not generalize.")


In [ ]:
def collect_test_fingerprints(model, paths):
    cache = {}
    summary_rows = []
    for image_index, path in enumerate(paths):
        image = load_image(path)
        boxes, scores = infer_np(model, image)
        keep = scores >= CFG["BASE_THR"]
        boxes = boxes[keep]
        scores = scores[keep]
        fingerprints = []
        for box, score in zip(boxes, scores):
            fingerprint = fingerprint_candidate(model, image, box, score)
            fingerprints.append([fingerprint[name] for name in FEATURE_NAMES])
        feature_array = np.asarray(fingerprints, np.float32).reshape(-1, len(FEATURE_NAMES))
        probabilities = score_fingerprint(feature_array, full_calibrator) if len(boxes) else np.zeros(0)
        cache[path.stem] = {
            "boxes": boxes,
            "scores": scores,
            "features": feature_array,
            "poison_probability": probabilities.astype(np.float32),
        }
        for candidate_index, (score, probability) in enumerate(zip(scores, probabilities)):
            summary_rows.append({
                "image_id": path.stem,
                "candidate_index": candidate_index,
                "teacher_score": float(score),
                "poison_probability": float(probability),
            })
        if (image_index + 1) % 100 == 0:
            print(f"test fingerprints | {image_index + 1}/{len(paths)}")
    return cache, pd.DataFrame(summary_rows)

test_cache, test_fingerprint_frame = collect_test_fingerprints(model, test_files)
test_fingerprint_frame.to_csv(WORK / "nb53_test_fingerprints.csv", index=False)
candidate_count = int(sum(len(item["scores"]) for item in test_cache.values()))
assert candidate_count == len(test_fingerprint_frame)
print(
    f"test | candidates={candidate_count} "
    f"p50={test_fingerprint_frame['poison_probability'].median():.4f} "
    f"p90={test_fingerprint_frame['poison_probability'].quantile(0.90):.4f}"
)


In [ ]:
def normalize_id(value):
    text = str(value)
    try:
        return str(int(text))
    except ValueError:
        return text

def format_prediction(boxes, scores):
    values = []
    for (x1, y1, x2, y2), score in zip(boxes, scores):
        x1 = float(np.clip(x1, 0, 1024)); y1 = float(np.clip(y1, 0, 1024))
        x2 = float(np.clip(x2, 0, 1024)); y2 = float(np.clip(y2, 0, 1024))
        width, height = x2 - x1, y2 - y1
        if width > 0 and height > 0 and score > 0:
            values.extend([
                f"{float(score):.6f}", f"{x1:.2f}", f"{y1:.2f}",
                f"{width:.2f}", f"{height:.2f}",
            ])
    return " ".join(values) if values else " "

def compose_response_submission(cache, strength, power, output_path):
    predictions = {}
    output_count = 0
    confidence_mass = 0.0
    for stem, item in cache.items():
        attenuation = np.power(
            np.clip(1.0 - strength * item["poison_probability"], 0.0, 1.0),
            power,
        )
        calibrated_scores = CFG["EPS"] + (
            item["scores"] - CFG["EPS"]
        ) * attenuation
        output_scores = np.maximum(CFG["EPS"], calibrated_scores)
        output_count += len(output_scores)
        confidence_mass += float(output_scores.sum())
        predictions[normalize_id(stem)] = (item["boxes"], output_scores)
    sample = pd.read_csv(SAMPLE_SUB, dtype={"image_id": str})
    sample["prediction_string"] = sample["image_id"].map(
        lambda image_id: format_prediction(*predictions.get(
            normalize_id(image_id),
            (np.zeros((0, 4), np.float32), np.zeros(0, np.float32)),
        ))
    )
    sample.to_csv(output_path, index=False)
    return output_count, confidence_mass

manifest_rows = []
if RESPONSE_GATES_PASS:
    for candidate in RESPONSE_CANDIDATES:
        output_path = WORK / OUTPUT_BY_NAME[candidate["name"]]
        output_count, confidence_mass = compose_response_submission(
            test_cache, candidate["strength"], candidate["power"], output_path
        )
        assert output_count == candidate_count
        manifest_rows.append({
            **candidate,
            "path": str(output_path),
            "candidate_count": candidate_count,
            "output_count": output_count,
            "confidence_mass": confidence_mass,
            "response_auc": response_auc,
            "response_partial_auc_10": response_partial_auc,
            "submit_first": candidate["priority"] == 1,
        })

manifest = pd.DataFrame(manifest_rows).sort_values("priority")
assert int(manifest["submit_first"].sum()) == 1
manifest.to_csv(WORK / "nb53_manifest.csv", index=False)
with open(WORK / "nb53_run_report.json", "w") as handle:
    json.dump({
        "mode": MODE,
        "response_gates_pass": RESPONSE_GATES_PASS,
        "response_auc": response_auc,
        "response_partial_auc_10": response_partial_auc,
        "response_tpr10": response_tpr10,
        "response_margin": response_margin,
        "fold_auc": fold_auc,
        "calibrator": full_calibrator,
        "candidates": manifest_rows,
    }, handle, indent=2)

print("\n=== NB53 MANIFEST ===")
print(manifest[[
    "name", "strength", "power", "candidate_count", "confidence_mass",
    "response_auc", "submit_first", "path",
]].to_string(index=False))
print("NEVER auto-submit: inspect the response audit, then submit CF_BALANCED first only.")
